# Corrected Preprocessing Pipeline

## Keputusan preprocessing

- `X` dan `y` dipisahkan sebelum pembagian data. `target_binary` menjadi `y`; `target` dan `source_file` dikeluarkan dari predictor untuk mencegah leakage.
- Data dibagi menggunakan stratified train-test split. Stratifikasi menjaga proporsi kelas pada train dan test.
- Semua transformer di dalam pipeline hanya di-fit pada training set. Dengan demikian median imputasi, kategori hasil encoding, dan parameter scaling tidak melihat test set.
- Missing values numerik diisi dengan median training, sedangkan missing values kategorikal diisi dengan modus training.
- Logistic Regression sensitif terhadap skala fitur numerik, sehingga `StandardScaler` digunakan pada fitur numerik. One-hot encoded features tidak diskalakan lagi.
- Tidak digunakan SMOTE, feature selection tambahan, maupun hyperparameter tuning. Model dan parameternya ditetapkan satu kali.
- Test set hanya digunakan untuk evaluasi final setelah pipeline selesai di-fit pada training set.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Pisahkan predictor dan target sebelum split.
# target_binary adalah label; target asli dan source_file tidak boleh masuk ke X.
X = df.drop(columns=["target", "target_binary", "source_file"]).copy()
y = df["target_binary"].astype(int).copy()

print(f"X shape sebelum split: {X.shape}")
print(f"y shape sebelum split: {y.shape}")
print("Distribusi y:")
display(y.value_counts(normalize=True).rename("proportion").to_frame())

# 2. Split dilakukan sebelum transformer dibuat dan di-fit.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Proporsi kelas train: {y_train.mean():.3f}")
print(f"Proporsi kelas test: {y_test.mean():.3f}")

# Kolom kategorikal diperlakukan sebagai kategori walaupun tersimpan sebagai angka.
numeric_features_model = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_features_model = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features_model),
        ("categorical", categorical_pipeline, categorical_features_model),
    ],
    remainder="drop",
)

# 3. Pipeline mem-fit preprocessing dan model hanya menggunakan training set.
# Tidak ada SMOTE, feature selection, atau tuning tambahan.
model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

model_pipeline.fit(X_train, y_train)
print("Pipeline berhasil di-fit pada training set.")

# Verifikasi bahwa parameter imputer berasal dari training set dan test belum di-fit.
trained_preprocessor = model_pipeline.named_steps["preprocessing"]
trained_numeric_imputer = trained_preprocessor.named_transformers_["numeric"].named_steps["imputer"]
print("Median imputasi training:")
display(pd.Series(trained_numeric_imputer.statistics_, index=numeric_features_model, name="training_median"))

## Final evaluation

Prediksi pada `X_test` dilakukan hanya setelah seluruh preprocessing dan model selesai di-fit dengan `X_train`. Tidak ada informasi dari test set yang digunakan untuk memilih atau mengubah parameter pipeline.

In [ ]:
# 4. Test set hanya dipakai untuk final evaluation.
y_test_pred = model_pipeline.predict(X_test)
y_test_proba = model_pipeline.predict_proba(X_test)[:, 1]

print("FINAL TEST EVALUATION")
print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_test_proba):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_test_pred, target_names=["No disease", "Disease"]))

confusion = pd.DataFrame(
    confusion_matrix(y_test, y_test_pred),
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"],
)
display(confusion)

print("Jumlah fitur setelah preprocessing:", len(model_pipeline.named_steps["preprocessing"].get_feature_names_out()))